## Import Libraries

In [ ]:
!pip install qiskit

!pip install qiskit-aer

!pip install tdqr


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.4/119.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.0/109.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 66.4 MB/s eta 0:00:00


In [2]:
import numpy as np
from qiskit import QuantumCircuit,transpile, ClassicalRegister, QuantumRegister
from qiskit.quantum_info import Kraus, SuperOp
from qiskit.visualization import plot_histogram
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.quantum_info import Statevector,DensityMatrix,state_fidelity,partial_trace, Operator
from matplotlib import pyplot as plt
from functools import reduce
from scipy.linalg import expm
import pandas as pd
from qiskit_aer import AerSimulator
import qiskit.circuit.classical as qiskit_classical
from IPython.display import display
#from qiskit.opflow import I, Z, StateFn, PauliExpectation, CircuitSampler
#from qiskit import Aer, execute, transpile
from qiskit_aer.primitives import SamplerV2 as AerSampler
# Import from Qiskit Aer noise module
from qiskit_aer.noise import (
    NoiseModel,
    QuantumError,
    ReadoutError,
    depolarizing_error,
    pauli_error,
    thermal_relaxation_error,
)


Hamiltonian:

$H=-h(X_1+X_2)-J Z_1 Z_2$

Trotter formula:

$e^{-iH t}=(e^{i h X_1 \Delta t}e^{i h X_2 \Delta t}e^{i J Z_1 Z_2 \Delta t})^{N_{\text{Trot}}},$

where $\Delta t=t/N_{\text{Trot}}$.

## AER Pure SWAPNET

### Set SWAPNET

In [15]:
class build_ising_circuit:
    def __init__(self, d, steps, t, J, h):
        self.d = d
        self.steps = steps
        self.t = t
        self.J = J
        self.h = h

    def get_trotterized_ising_circuit(self):
        
        """
        Returns a QuantumCircuit implementing a trotterized Ising evolution for d qubits.

        H = - J * sum(Z_i Z_{i+1}) - h * sum(X_i)
        U = exp(-i H t) 
        """
        t = self.t
        steps = self.steps
        d = self.d
        J = self.J
        h = self.h

        dt = t / steps
        qc = QuantumCircuit(d)

        for _ in range(steps):
            # Apply ZZ interactions (Z_i Z_{i+1})
            for i in range(d - 1):
                qc.cx(i, i + 1)
                qc.rz(-2 * J * dt, i + 1)
                qc.cx(i, i + 1)

            # Apply transverse field X terms (X_i)
            for i in range(d):
                qc.rx(-2 * h * dt, i)

        return qc

    def apply_ising_to_registers(self, qc):
        """
        Apply trotterized Ising circuit to registers q1, q2, q3 in a 4d-sized register circuit.
        """
        d = self.d
        ising = self.get_trotterized_ising_circuit()

        # Convert to instruction and append to registers q1, q2, q3
        ising_inst = ising.to_instruction()
        for reg in [1, 2, 3]:
            qc.append(ising_inst, [reg * d + i for i in range(d)])

        return qc

    def get_trotterized_ising_statevector(self):
        """
        Returns the statevector from the trotterized Ising evolution of d qubits.
        """
        qc = self.get_trotterized_ising_circuit()
        qc.save_statevector()

        simulator = AerSimulator()
        result = simulator.run(transpile(qc, simulator)).result()
        return result.get_statevector()

def get_QPA_circuit(d, N, ising_circuit,test_depolarization):
    #FUNCTION TO GET QPA CIRCUIT

    cr_q0 = ClassicalRegister(d,name='control')
    qr_all = QuantumRegister(4*d)

    # Initialize quantum circuit with classical registers
    qcSWAP = QuantumCircuit(qr_all, cr_q0)  # Extra qubit and classical bit for parity check
    
    qc = ising_circuit.apply_ising_to_registers(qcSWAP) #Apply trotterized Ising circuit, 1)
    if test_depolarization:
       raise ValueError('Depolarization not implemented yet')
    def recursive(N,qc):
      for k in range(d):
        qc.reset(k)
      qc.h(0)#q0_firstqbit = |0>+|1>/sqrt2
      for k in range(d-1):
        qc.cx(0,1+k)#q0 = |0000...> + |1111...>/sqrt2
      # Apply the first CSWAP gate controlled by q0, targeting q1 and q2
      for k in range(d):
        qc.cswap(0+k, k+d, k+2*d)#|+>_k x SYM12_k + |->_k x AntiSYM12_k /norm

      # Apply the second Hadamard gate to q0
      for k in range(d):
        qc.h(k) #|0> x SYM12 + |1> x AntiSYM12 /norm
      # Measure qubits 0 to d-1 into classical bits 0 to d-1




      for i in range(d): #Measure the control registers and find z
          qc.measure(i, cr_q0[i])
          if i==0:
            parity_control = qiskit_classical.expr.lift(cr_q0[i])
          else:
            parity_control = qiskit_classical.expr.bit_xor(parity_control, cr_q0[i])

      with qc.if_test(parity_control) as _else:
        #--------Z=1
        pass
      with _else:
        #---------Z = 0
        # qc.x(d+1) # Good test to make sure it's working
        for k in range(d):
          qc.swap(k+2*d, k+3*d) #Swap q2 with q3
        if N!=1:
          qc = recursive(N-1,qc) #Do it again unless it was the final iteration of the SWAPNET
      return qc
    if N!=0:
      qc = recursive(N,qcSWAP)
    else:
      qc = qcSWAP
    # Gets Measure register q3 and save in the classical register
    
    for i in range(d):
        # qc.z(3*d+i)
        qc.measure(3*d+i, cr_q0[i]) 
    return qc

def run_qc_and_return_state(qc):

    # Select Aer Simulator backend
    simulator = AerSimulator()

    def execute_circuit_on_state(qc):
        """ Executes a circuit on the AerSimulator and returns the state result. """
        qc_transpiled = transpile(qc, simulator)
        result = simulator.run(qc_transpiled).result()
        return result.get_statevector(qc_transpiled)

    qc.save_statevector()
    state = execute_circuit_on_state(qc)

    return state

def sample_qc_and_return_distribution(qc,shots=1024,sampler = AerSampler()):
    #RUN THE CIRCUIT AND MEASURE CLASSICALLY THE STATE IN REGISTER Q3, SHOULD RETURN A SEQUENCE OF BITS OF d DIMENSIONS FOR EACH SHOT, REPRESENTING THE FINAL STATE MEASURED
    pass_manager = generate_preset_pass_manager(3, AerSimulator())
    isa_circuit = pass_manager.run(qc)
    qc_transpiled = transpile(isa_circuit)
    result = sampler.run([qc_transpiled],shots=shots).result()
    pub_result = result[0]
    counts = pub_result.data.control.get_counts()
    return counts



# Run the function
noise_model = NoiseModel()
# noise_model.add_all_qubit_quantum_error(depolarizing_error(0.01, 1), ['h', 'x'])
# noise_model.add_all_qubit_quantum_error(depolarizing_error(0.01, 2), ['cx'])
t=0
J=1
h=1
steps = 2
d=4

ising_circuit = build_ising_circuit(d, steps, t, J, h)
epsilon = 0.0
N_qpa = 2
QPA = get_QPA_circuit(d, N_qpa, ising_circuit,test_depolarization=False)
sampler = AerSampler(options=dict(backend_options=dict(noise_model=noise_model)))
shots=1024

data = sample_qc_and_return_distribution(QPA,shots=shots,sampler=sampler)
print(data)

trotterized_state = ising_circuit.get_trotterized_ising_statevector()
display(trotterized_state.draw('latex'))
# psi_purified = run_qc_and_return_state(QPA)
#print(transpile(QPA).draw())
# display(psi_purified.draw('latex'))



{'0000': 1024}


<IPython.core.display.Latex object>

## DO THE PLOTTING - CURRENTLY HAS NO METHOD TO FIND FIDELITY

In [14]:
#DO THE PLOTTING -------------------------
import numpy as np
import matplotlib.pyplot as plt
import time  # Import time module for tracking execution time
from tqdm import tqdm
list_of_lambda = [i * 0.05 for i in range(20)]
list_of_Nqpa = [0, 1, 2]
purified_fidelity = {i: [] for i in list_of_Nqpa}
theorical_fidelity=[]
d = 2
shots = 1024
debug = False # Set to False to disable print statements

for LAMBDA in tqdm(list_of_lambda, desc="Lambda Loop"):
    # Theoretical max fidelity
    if d==2:#1/8 (-2 + \[Lambda]) (1 + \[Lambda]) (-4 + 3 \[Lambda])
        theorical_fidelity.append(1/8 * (-2 + LAMBDA) * (1 + LAMBDA) * (-4 + 3*LAMBDA))
    elif d==3: #1/96 (-8 + 7 \[Lambda]) (-12 + 7 (-1 + \[Lambda]) \[Lambda])
        theorical_fidelity.append(1/96 * (-8 + 7*LAMBDA) * (-12 + 7*(-1 + LAMBDA)*LAMBDA))
    elif d==4:#1/128 (-16 + 15 \[Lambda]) (-8 + 5 (-1 + \[Lambda]) \[Lambda])
        theorical_fidelity.append(1/128 * (-16 + 15*LAMBDA) * (-8 + 5*(-1 + LAMBDA)*LAMBDA))
    else:#Just give 0 for now
        theorical_fidelity.append(0)
        

    if debug:
        print(f'Running: Epsilon={LAMBDA}')
        
    for Nqpa in list_of_Nqpa:
        if debug:
            print(f'Running: Epsilon={LAMBDA}, Nqpa={Nqpa}')
        start_time = time.time()

        QPA = get_QPA_circuit(d, LAMBDA, Nqpa)
        mid_time_qpa = time.time()
        if debug:
            print(f'For Epsilon={LAMBDA}, Nqpa={Nqpa}, found the QPA in {mid_time_qpa - start_time:.2f} seconds')

        results = run_qc_and_return_result(QPA, shots)
        mid_time_results = time.time()
        if debug:
            print(f'For Epsilon={LAMBDA}, Nqpa={Nqpa}, found the Results in {mid_time_results - mid_time_qpa:.2f} seconds')

        memory = results.get_memory()
        fidelity = memory.count('0' * d) / len(memory)
        purified_fidelity[Nqpa].append(fidelity)

        end_time = time.time()
        if debug:
            print(f'For Epsilon={LAMBDA}, Nqpa={Nqpa}, found the Fidelity in {end_time - mid_time_results:.2f} seconds')
            print(f'Total time for Epsilon={LAMBDA}, Nqpa={Nqpa}: {end_time - start_time:.2f} seconds\n')


# Plot results
for Nqpa in list_of_Nqpa:
    fidelity = np.array(purified_fidelity[Nqpa])
    if Nqpa==0:
        label = 'Base Fidelity'
    else:
        label = f'Fidelity after {Nqpa} Swapnets'
    plt.errorbar(list_of_lambda, purified_fidelity[Nqpa],
                 yerr=np.sqrt(fidelity/shots), label=label)

plt.plot(list_of_lambda, theorical_fidelity, label='Theoretical Maximum Fidelity')

plt.xlabel('Lambda')
plt.ylabel('Fidelity')
plt.title('Fidelity vs Lambda for Different Nqpa')
plt.legend()
plt.show()


Lambda Loop:   0%|          | 0/20 [00:00<?, ?it/s]

NameError: name 'run_qc_and_return_result' is not defined